# Board Registry — Runtime-Registrierung von Board-Typen

Dieses Notebook demonstriert das **Kernfeature** des Board-Registry-Service (Port 8007):
neue Board-*Typen* zur **Laufzeit** über die öffentliche REST-API zu registrieren — ohne
Redeploy und ohne Datenbankmigration.

**Ausgangslage (out of the box):** Der Service liefert zwei eingebaute (`built_in`) Typen
mit, die per Baseline-Migration (`0001_init`) angelegt werden:

| Typ | View | Zweck |
|-----|------|-------|
| `kanban` | board | Spalten-Board mit WIP-Limit |
| `calendar` | calendar | Termin-/Datumsansicht |

Eingebaute Typen sind **unveränderlich** (nicht editier-/löschbar). In diesem Notebook
registrieren wir zusätzlich **`scrum`** und **`gantt`** über die API. Diese sind dann
*normale* (nicht eingebaute) Typen und damit voll editier- und löschbar — genau das ist
der Punkt: jeder authentifizierte Nutzer kann den Katalog erweitern.

> Hintergrund zur deklarativen `presentation`-Spec: siehe `docs/decisions/0002-board-view-extensibility.md`
> und `docs/services/boardregistry.md`.

## Voraussetzungen

1. Der Stack läuft und ist *healthy*:
   ```bash
   make up
   ```
2. Die eingebauten Typen `kanban` und `calendar` existieren (durch die Migration).
3. `scrum`/`gantt` sind **noch nicht** registriert. Bei einer frisch aufgesetzten DB
   (`make clean-volumes && make up`) ist das der Fall. Sind sie schon vorhanden,
   meldet die API `409 board_type_exists` — das Notebook behandelt das sauber.
4. Ein Nutzer-Account zum Einloggen. Wir verwenden den Demo-User `alice` und legen ihn
   bei Bedarf an (identisch zu `make seed`).

Es werden nur Standardbibliotheken verwendet (`urllib`), keine zusätzlichen Pakete nötig.

In [28]:
import json
import os
import urllib.error
import urllib.request

# Das Gateway (Traefik) routet /api/v1/board-types an den Board-Registry-Service.
API = os.environ.get("API_BASE_URL", "http://localhost:3000")
ALICE = {"email": "alice@teamboard.local", "password": os.environ.get("SEED_ALICE_PASSWORD", "AliceSecret123!")}


def call(method, path, body=None, token=None):
    """Minimaler JSON-HTTP-Client. Gibt (status_code, parsed_json) zurück."""
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(f"{API}{path}", data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req) as resp:
            return resp.status, json.loads(resp.read() or "null")
    except urllib.error.HTTPError as e:
        raw = e.read()
        try:
            return e.code, json.loads(raw)
        except Exception:
            return e.code, {"body": raw.decode(errors="replace")}


print(f"API base: {API}")

API base: http://localhost:3000


## 1. Authentifizieren

Schreibzugriffe auf den Katalog erfordern ein gültiges JWT (aktuell: jeder authentifizierte
Nutzer darf registrieren — eine Beschränkung auf eine Admin-/Publisher-Rolle ist als
Folgearbeit vorgesehen, siehe ADR 0001). Wir registrieren `alice` (idempotent) und loggen
uns ein.

In [29]:
# Registrierung ist idempotent: ein 409 (bereits vorhanden) ist ok.
status, _ = call("POST", "/api/v1/auth/register", ALICE)
print("register:", status, "(409 = existiert bereits, ebenfalls ok)")

status, login = call("POST", "/api/v1/auth/login", ALICE)
assert status == 200, f"Login fehlgeschlagen: {status} {login}"
TOKEN = login["data"]["access_token"]
print("login: ok, Token erhalten")

register: 409 (409 = existiert bereits, ebenfalls ok)
login: ok, Token erhalten


## 2. Aktuellen Katalog ansehen

Vor der Registrierung sollten nur die eingebauten Typen `kanban` und `calendar` erscheinen.

In [17]:
def print_catalog():
    status, resp = call("GET", "/api/v1/board-types", token=TOKEN)
    assert status == 200, f"{status} {resp}"
    rows = resp["data"]
    print(f"{len(rows)} Board-Typ(en):")
    for d in rows:
        flag = "built_in" if d["built_in"] else "custom  "
        print(f"  [{flag}] {d['type']:<10} {d['icon']} {d['display_name']:<18} view={d['presentation'].get('view', 'board')}")


print_catalog()

4 Board-Typ(en):
  [built_in] calendar   📅 Calendar           view=calendar
  [built_in] kanban     📋 Kanban Board       view=board
  [custom  ] gantt      📊 Gantt (Timeline)   view=timeline
  [custom  ] scrum      🏃 Scrum Board        view=board


## 3. Anatomie einer Board-Typ-Definition

Eine Definition besteht aus:

- **`type`** — Slug (`^[a-z][a-z0-9_-]{0,49}$`), eindeutig.
- **`display_name`**, **`icon`** — Anzeige im Frontend.
- **`default_columns`** — Spalten, die ein neues Board dieses Typs erhält. Jede Spalte trägt
  einen expliziten semantischen **`status`** (`open` | `in_progress` | `blocked` | `done` |
  `archived`), den der Task-Service direkt übernimmt (statt ihn aus dem Namen zu raten).
- **`default_config`** — typ-spezifische Default-Konfiguration eines Boards.
- **`config_schema`** — JSON-Schema, gegen das die Board-Config validiert wird (frei vom
  Typ-Autor definiert).
- **`presentation`** — deklarative Rendering-Hints (gegen ein **host-definiertes** Meta-Schema
  validiert): `view` (`board` | `calendar` | `timeline`), `view_config` (je View erlaubte Keys)
  und `card` (Felder + `color_by`).

### 3a. `scrum` — Board-View mit eigenem `config_schema`

Scrum nutzt denselben `board`-Renderer wie Kanban, bringt aber mehr Default-Spalten und ein
`config_schema` mit, das `sprint_length_days` (1–90) validiert.

In [9]:
scrum = {
    "type": "scrum",
    "display_name": "Scrum Board",
    "icon": "\U0001F3C3",
    "default_columns": [
        {"name": "Backlog", "position": 0, "status": "open"},
        {"name": "Sprint", "position": 1, "status": "open"},
        {"name": "In Progress", "position": 2, "status": "in_progress"},
        {"name": "Review", "position": 3, "status": "in_progress"},
        {"name": "Done", "position": 4, "status": "done"},
    ],
    "default_config": {"sprint_length_days": 14},
    "config_schema": {
        "type": "object",
        "properties": {"sprint_length_days": {"type": "integer", "minimum": 1, "maximum": 90}},
        "additionalProperties": False,
    },
    "presentation": {
        "view": "board",
        "view_config": {"group_by": "column", "show_wip": True},
        "card": {
            "fields": ["priority", "due_date", "labels", "comment_count", "attachment_count"],
            "color_by": "priority",
        },
    },
}


def register(definition):
    status, resp = call("POST", "/api/v1/board-types", body=definition, token=TOKEN)
    if status == 201:
        print(f"  registriert: {definition['type']} (built_in={resp['data']['built_in']})")
    elif status == 409:
        print(f"  {definition['type']}: existiert bereits (409) — übersprungen")
    else:
        print(f"  FEHLER {status}: {resp}")
    return status, resp


register(scrum);

  registriert: scrum (built_in=False)


### 3b. `gantt` — der `timeline`-Renderer

Gantt demonstriert eine **andere View**: `timeline`. Die `view_config` zeigt auf die
Task-Felder, die die Balken aufspannen (`start_field` = `start_date`, `end_field` = `due_date`)
und färbt nach Priorität. Genau hierfür existiert das optionale `tasks.start_date`-Feld.

In [20]:
gantt = {
    "type": "gantt",
    "display_name": "Gantt (Timeline)",
    "icon": "\U0001F4CA",
    "default_columns": [
        {"name": "Planned", "position": 0, "status": "open"},
        {"name": "In Progress", "position": 1, "status": "in_progress"},
        {"name": "Done", "position": 2, "status": "done"},
    ],
    "default_config": {},
    "config_schema": {},
    "presentation": {
        "view": "timeline",
        "view_config": {
            "start_field": "start_date",
            "end_field": "due_date",
            "group_by": "column",
            "color_by": "priority",
        },
        "card": {"fields": ["priority", "due_date"], "color_by": "priority"},
    },
}

register(gantt);

  registriert: gantt (built_in=False)


## 4. Ergebnis prüfen

Der Katalog enthält jetzt vier Typen: zwei eingebaute und zwei zur Laufzeit registrierte.
Der Project-Service bezieht diese Definitionen über die interne API und invalidiert seinen
Cache auf die `boardtype.registered`-Events, die diese POSTs ausgelöst haben — neue Boards
dieser Typen sind also sofort anlegbar.

In [ ]:
print_catalog()

## 5. Validierung demonstrieren (optional)

Die `presentation`-Spec wird gegen ein host-definiertes Meta-Schema geprüft. Ein unbekannter
`view`-Wert wird mit `400 validation_failed` abgelehnt — so kann ein Typ-Autor nur Renderer
wählen, die das Frontend tatsächlich mitbringt.

In [18]:
bad = {
    "type": "spreadsheet-demo",
    "display_name": "Invalid View",
    "default_columns": [{"name": "A", "position": 0, "status": "open"}],
    "presentation": {"view": "spreadsheet"},  # nicht erlaubt
}
status, resp = call("POST", "/api/v1/board-types", body=bad, token=TOKEN)
print(status, resp.get("code"), "-", resp.get("detail"))

400 validation_failed - presentation.view must be one of board, calendar, timeline


## 6. Lebenszyklus: editieren / löschen (optional)

Da `scrum`/`gantt` **nicht** eingebaut sind (`built_in = false`), lassen sie sich per `PATCH`
ändern und per `DELETE` entfernen. Bei den eingebauten `kanban`/`calendar` würde dies mit
`403 builtin_immutable` scheitern.

Die folgende Zelle ist auskommentiert, damit das Notebook standardmäßig nichts wieder
abräumt — bei Bedarf einkommentieren.

In [19]:
# PATCH: Anzeigename ändern
status, resp = call("PATCH", "/api/v1/board-types/scrum", body={"display_name": "Scrum (Team A)"}, token=TOKEN)
print("patch scrum:", status, resp["data"]["display_name"] if status == 200 else resp)

# DELETE: Typ wieder entfernen
status, _ = call("DELETE", "/api/v1/board-types/gantt", token=TOKEN)
print("delete gantt:", status, "(204 = gelöscht)")

# Gegenprobe: eingebauten Typ löschen schlägt fehl
status, resp = call("DELETE", "/api/v1/board-types/kanban", token=TOKEN)
print("delete kanban:", status, resp.get("code"))

patch scrum: 200 Scrum (Team A)
delete gantt: 204 (204 = gelöscht)
delete kanban: 403 builtin_immutable
